# Fine-tune Qwen3.5-9B (LoRA) cho lớp data DataMate

Dữ liệu: `train.jsonl`, `val.jsonl` (tạo bằng `python -m datamate.finetune.make_data`, nằm ở `solve data/finetune/`).
Mỗi dòng là một cuộc chat system / user / assistant, đúng y prompt pipeline gửi khi chạy thật.

**Trước khi chạy:** tắt server vLLM đang chạy (nó giữ gần hết bộ nhớ GPU), cần GPU A100.
Chạy lần lượt từng ô. Xong thì adapter nằm ở Google Drive `MyDrive/datamate_lora`, rồi bật lại vLLM theo ô cuối.

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
# KHÔNG nâng cấp transformers: giữ bản vLLM đang dùng (nâng cấp có thể làm vỡ vLLM khi bật lại)
!pip -q install peft accelerate
!pip uninstall -y torchao   # bản cũ làm peft báo lỗi; không dùng tới
import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os
os.makedirs('/content/ft', exist_ok=True)
# Tải 2 file train.jsonl, val.jsonl lên (hoặc chép sẵn vào Drive rồi sửa đường dẫn dưới)
up = files.upload()
for k, v in up.items():
    open(f'/content/ft/{k}', 'wb').write(v)
!wc -l /content/ft/*.jsonl

In [ ]:
import json, torch
from transformers import AutoConfig, AutoTokenizer
MODEL = 'Qwen/Qwen3.5-9B'
try:
    cfg = AutoConfig.from_pretrained(MODEL)
except Exception as e:
    raise SystemExit(f'transformers nay chua doc duoc {MODEL}: {e}. Khi do moi chay: pip install -U transformers roi Restart session')
print('architectures:', cfg.architectures, '| model_type:', cfg.model_type)
tok = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
# Mã hoá: chỉ tính loss trên phần trả lời (assistant). Prompt dựng giống vLLM lúc chạy: tắt chế độ suy nghĩ.
MAX_LEN = 4096
# kết thúc lượt chat: vLLM dừng sinh ở token này
END = '<|im_end|>' if '<|im_end|>' in tok.get_vocab() else tok.eos_token
def encode(ex):
    msgs = ex['messages']
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    full = prompt + msgs[-1]['content'] + END
    p = tok(prompt, add_special_tokens=False)['input_ids']
    f = tok(full, add_special_tokens=False)['input_ids']
    assert f[:len(p)] == p, 'prompt không phải tiền tố của bài đầy đủ'
    f = f[:MAX_LEN]
    return {'input_ids': f, 'labels': ([-100] * len(p) + f[len(p):])[:len(f)]}

def load(path):
    return [encode(json.loads(l)) for l in open(path, encoding='utf-8')]
train_ds, val_ds = load('/content/ft/train.jsonl'), load('/content/ft/val.jsonl')
lens = sorted(len(x['input_ids']) for x in train_ds)
print(len(train_ds), len(val_ds), 'token: trung vị', lens[len(lens)//2], 'max', lens[-1])
print(tok.decode(train_ds[0]['input_ids'])[-400:])

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')
except Exception as e:      # bản đa phương thức: nạp lớp có ảnh, LoRA chỉ trên phần ngôn ngữ
    print('AutoModelForCausalLM lỗi:', e)
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules='all-linear', exclude_modules=r'.*(visual|vision|mtp).*')
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
args = TrainingArguments(
    output_dir='/content/ft_out', per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, num_train_epochs=2, learning_rate=1e-4, lr_scheduler_type='cosine',
    warmup_steps=12, logging_steps=10, eval_strategy='steps', eval_steps=60, save_strategy='no',
    bf16=True, report_to='none', gradient_checkpointing=True)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()
OUT = '/content/drive/MyDrive/datamate_lora'
model.save_pretrained(OUT); tok.save_pretrained(OUT)
print('đã lưu', OUT)

In [ ]:
# Thử nhanh một câu của tập kiểm
model.eval()
ex = json.loads(open('/content/ft/val.jsonl', encoding='utf-8').readline())
prompt = tok.apply_chat_template(ex['messages'][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
ids = tok(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=300, do_sample=False)
print('model :', tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
print('đáp án:', ex['messages'][-1]['content'])

## Bật lại vLLM kèm adapter

Giải phóng GPU (Runtime → Restart session), rồi chạy lại ô bật vLLM cũ của bạn, **thêm 3 tham số**:

```
--enable-lora --lora-modules datamate=/content/drive/MyDrive/datamate_lora --max-lora-rank 16
```

Model gốc vẫn gọi được bằng tên `Qwen/Qwen3.5-9B` (phía query dùng như cũ). Lớp data gọi adapter bằng tên `datamate`
(`DATAMATE_LLM_MODEL=datamate`). Bật cloudflared như cũ rồi gửi link mới.

Nếu vLLM báo không hỗ trợ LoRA cho model này: chạy ô dưới để gộp adapter vào model rồi phục vụ thư mục đã gộp.

In [ ]:
# (chỉ khi vLLM không nhận LoRA) gộp adapter vào model, lưu bản đầy đủ
merged = model.merge_and_unload()
merged.save_pretrained('/content/drive/MyDrive/datamate_merged', safe_serialization=True)
tok.save_pretrained('/content/drive/MyDrive/datamate_merged')
# rồi: vllm serve /content/drive/MyDrive/datamate_merged --served-model-name datamate ...